<a href="https://colab.research.google.com/github/Jeffrey1999/Deep-Learning/blob/main/ATTENTIVESECURE1_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================
# ATTENTIVESECURE v5: HIGH ACCURACY (Target: ≥94.5%)
# ===============================

from google.colab import drive
drive.mount('/content/drive')

!pip install -q scikit-learn tensorflow matplotlib seaborn

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# === CONFIG ===
DATA_PATH = "/content/drive/MyDrive/Datasets/Edge-IIoTset dataset/Selected dataset for ML and DL/DNN-EdgeIIoT-dataset.csv"
LABEL_COLUMN = 'Attack_type'
SAMPLE_SIZE = 1_500_000  # ↑↑↑ Increased from 500k

# === 1. Load Data ===
print("Loading dataset...")
df = pd.read_csv(DATA_PATH, low_memory=False)

if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
    print(f"Stratified sampling to {SAMPLE_SIZE:,} rows...")
    df = df.groupby(LABEL_COLUMN, group_keys=False).apply(
        lambda x: x.sample(min(len(x), int(SAMPLE_SIZE * len(x) / len(df))), random_state=42)
    ).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")

# === 2. Preprocess ===
X = df.drop(columns=[LABEL_COLUMN])
y = df[LABEL_COLUMN]

X = X.select_dtypes(include=[np.number]).astype(np.float32)
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)

# ✅ Use RobustScaler (better than MinMax for outliers)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

X_reshaped = X_scaled.reshape((X_scaled.shape[0], 1, X_scaled.shape[1]))
y_categorical = tf.keras.utils.to_categorical(y_encoded, num_classes=num_classes)

X_train, X_test, y_train, y_test = train_test_split(
    X_reshaped, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded
)

# === 3. Focal Loss (Tuned for Edge-IIoTset) ===
def focal_loss(gamma=1.0, alpha=0.75):
    def loss(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        return tf.reduce_mean(alpha_t * tf.pow(1 - p_t, gamma) * -tf.math.log(p_t))
    return loss

# === 4. Model: Deeper CNN + Optimized Transformer ===
def se_block(x, ratio=8):
    filters = x.shape[-1]
    se = tf.keras.layers.GlobalAveragePooling1D()(x)
    se = tf.keras.layers.Dense(filters // ratio, activation='relu')(se)
    se = tf.keras.layers.Dense(filters, activation='sigmoid')(se)
    se = tf.keras.layers.Reshape((1, filters))(se)
    return tf.keras.layers.Multiply()([x, se])

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.15):  # ↓ dropout
        super().__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([tf.keras.layers.Dense(ff_dim, activation="relu"), tf.keras.layers.Dense(embed_dim)])
        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = tf.keras.layers.Dropout(rate)
        self.drop2 = tf.keras.layers.Dropout(rate)
    def call(self, x, training=None):
        x = self.ln1(x + self.drop1(self.att(x, x), training=training))
        return self.ln2(x + self.drop2(self.ffn(x), training=training))

def build_model(input_dim, num_classes):
    inp = tf.keras.Input(shape=(1, input_dim))
    # ↑↑↑ Deeper CNN (like sSecure Net)
    x = tf.keras.layers.Conv1D(64, 1, activation='relu')(inp)
    x = tf.keras.layers.Conv1D(64, 1, activation='relu')(x)  # ← Second layer
    x = tf.keras.layers.Dropout(0.15)(x)  # ↓ dropout
    x = se_block(x)
    x = tf.keras.layers.Dense(64)(x)  # match embed_dim
    x = x + tf.Variable(tf.zeros((1, 1, 64)), trainable=False)
    x = TransformerBlock(64, 4, 128)(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.15)(x)
    out = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out)

# === 5. Train ===
model = build_model(X_train.shape[2], num_classes)
model.compile(optimizer='adam', loss=focal_loss(gamma=1.0, alpha=0.75), metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5)
]

print("\n🚀 Training High-Accuracy AttentiveSecure...")
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=256,  # ↑ batch size for stability
    callbacks=callbacks,
    verbose=1
)

# === 6. Evaluate ===
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

acc = accuracy_score(y_true_classes, y_pred_classes)
print(f"\n🎯 FINAL ACCURACY: {acc:.4f}")

print("\n📋 Classification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=le.classes_, digits=4))

# Save
model.save('/content/new-attentive_secure_high_acc.h5')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading dataset...
Stratified sampling to 1,500,000 rows...


/tmp/ipython-input-396465843.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(LABEL_COLUMN, group_keys=False).apply(


Dataset shape: (1499993, 63)

🚀 Training High-Accuracy AttentiveSecure...
Epoch 1/15
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 232s 60ms/step - accuracy: 0.7686 - loss: 0.0423 - val_accuracy: 0.7837 - val_loss: 0.0415 - learning_rate: 0.0010
Epoch 2/15
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 224s 60ms/step - accuracy: 0.7720 - loss: 0.0425 - val_accuracy: 0.7825 - val_loss: 0.0410 - learning_rate: 0.0010
Epoch 3/15
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 218s 58ms/step - accuracy: 0.7682 - loss: 0.0428 - val_accuracy: 0.7814 - val_loss: 0.0411 - learning_rate: 0.0010
Epoch 4/15
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 263s 58ms/step - accuracy: 0.5552 - loss: nan - val_accuracy: 0.0113 - val_loss: nan - learning_rate: 0.0010
Epoch 5/15
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 261s 58ms/step - accuracy: 0.0111 - loss: nan - val_accuracy: 0.0113 - val_loss: nan - learning_rate: 0.0010
Epoch 6/15
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 216s 58ms/step - accuracy: 0.0112 - loss: nan - val_accuracy: 0.0113 - val_loss: nan - learning_rate: 5.0000e-0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



🎯 FINAL ACCURACY: 0.7831

📋 Classification Report:
                       precision    recall  f1-score   support

             Backdoor     0.0000    0.0000    0.0000      3361
            DDoS_HTTP     0.8913    0.0061    0.0121      6747
            DDoS_ICMP     0.0000    0.0000    0.0000     15740
             DDoS_TCP     0.0000    0.0000    0.0000      6768
             DDoS_UDP     0.9927    0.9977    0.9952     16434
       Fingerprinting     0.0000    0.0000    0.0000       135
                 MITM     0.0000    0.0000    0.0000       164
               Normal     0.7718    0.9986    0.8707    218409
             Password     0.0000    0.0000    0.0000      6780
        Port_Scanning     0.0000    0.0000    0.0000      3050
           Ransomware     0.0000    0.0000    0.0000      1477
        SQL_injection     0.0000    0.0000    0.0000      6922
            Uploading     0.0000    0.0000    0.0000      5087
Vulnerability_scanner     0.0000    0.0000    0.0000      6774
  